# Small vs Big : Qwen3 / Qwen2.5 vs Grok4, GPT-4.5, Llama 4 Behemoth, Gemini
Comparaison focalisée sur les modèles demandés : tailles (paramètres), compute, coût, énergie (quand dispo) et limites de couverture.


**Note** : Les CSV ne fournissent pas de métriques de performance (qualité). Les champs coût/énergie sont souvent manquants pour Qwen3/Qwen2.5; les graphiques indiquent les lacunes pour éviter de sur-interpréter.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.facecolor"] = "#f7f7f7"
plt.rcParams["figure.facecolor"] = "#f7f7f7"

DATA_DIR = Path("..") / "data" / "ai_models"


In [ ]:
def parse_number(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    s = str(value).lower()
    s = s.replace(",", "").replace(" ", "").replace("~", "").replace("≈", "")
    s = s.replace("×10^", "e").replace("x10^", "e").replace("×10", "e").replace("x10", "e")
    s = s.replace("^", "e").replace(">", "").replace("<", "")
    m = re.match(r"([0-9.+\-e]+)([kmbt]?)", s)
    if not m:
        return np.nan
    num, suf = m.groups()
    try:
        base = float(num)
    except ValueError:
        return np.nan
    mult = {"k": 1e3, "m": 1e6, "b": 1e9, "t": 1e12}.get(suf, 1)
    return base * mult


In [ ]:
files = {
    "frontier": pd.read_csv(DATA_DIR / "frontier_ai_models.csv"),
    "all": pd.read_csv(DATA_DIR / "all_ai_models.csv"),
    "notable": pd.read_csv(DATA_DIR / "notable_ai_models.csv"),
}

wanted_big = ["grok 4", "gpt-4.5", "llama 4 behemoth", "gemini"]
wanted_small = ["qwen3", "qwen 3", "qwen2.5", "qwen 2.5"]
pattern_big = re.compile("|".join(re.escape(w) for w in wanted_big), re.IGNORECASE)
pattern_small = re.compile("|".join(re.escape(w) for w in wanted_small), re.IGNORECASE)


def normalize(df, src):
    df = df.copy()
    df["publication_year"] = pd.to_datetime(df.get("Publication date"), errors="coerce").dt.year
    mapping = {
        "Training compute (FLOP)": "compute_flop",
        "Parameters": "parameters",
        "Training dataset size (gradients)": "train_tokens",
        "Training compute cost (2023 USD)": "train_cost_usd",
        "Training power draw (W)": "power_w",
        "Training time (hours)": "train_hours",
    }
    for src_col, tgt in mapping.items():
        df[tgt] = df[src_col].apply(parse_number) if src_col in df.columns else np.nan
    df["segment"] = src
    return df

normalized = {name: normalize(df, name) for name, df in files.items()}

small_df = pd.concat([
    n[n["Model"].fillna("").str.contains(pattern_small)]
    for n in normalized.values()
], ignore_index=True)
small_df = small_df.drop_duplicates(subset=["Model"], keep="first")
small_df["segment"] = "Qwen (2.5/3)"

big_df = pd.concat([
    normalized["frontier"],  # frontier contient Grok/GPT-4.5/Llama4 Behemoth/Gemini 1.0/1.5
    normalized["all"][normalized["all"]["Model"].fillna("").str.contains(pattern_big)],
    normalized["notable"][normalized["notable"]["Model"].fillna("").str.contains(pattern_big)],
], ignore_index=True)
# garder noms ciblés
big_df = big_df[big_df["Model"].fillna("").str.contains(pattern_big)].drop_duplicates(subset=["Model"], keep="first")
big_df["segment"] = "Frontier (Grok/GPT/Llama/Gemini)"

cols = ["Model", "Organization", "publication_year", "parameters", "compute_flop", "train_cost_usd", "power_w", "train_hours", "train_tokens", "segment"]
combined = pd.concat([small_df[cols], big_df[cols]], ignore_index=True)
combined


Couverture : Qwen2.5/Qwen3 entrent surtout via `all_ai_models`/`notable`; compute/cost/énergie sont souvent vides. Les gros modèles (Grok4, GPT‑4.5, Llama 4 Behemoth, Gemini 1.x/1.5) ont plus d'infos sur compute et coût; seule Gemini 1.0 Ultra expose puissance+durée.


In [ ]:
# Boxplot des paramètres
fig, ax = plt.subplots()
combined.boxplot(column="parameters", by="segment", ax=ax)
ax.set_yscale("log")
ax.set_ylabel("Paramètres")
ax.set_title("Taille des modèles : Qwen vs Big")
plt.suptitle("")
plt.tight_layout()
plt.show()


Les Qwen2.5/3 listés sont nettement plus petits que les modèles Grok/GPT/Llama/Gemini, avec plusieurs ordres de grandeur d'écart.


In [ ]:
# Compute vs paramètres (points disponibles)
cp = combined.dropna(subset=["parameters", "compute_flop"])
fig, ax = plt.subplots()
for seg, df_seg in cp.groupby("segment"):
    ax.scatter(df_seg["parameters"], df_seg["compute_flop"], label=seg, alpha=0.8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Paramètres")
ax.set_ylabel("Compute (FLOP)")
ax.set_title("Compute vs paramètres")
ax.legend()
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()


Les points Big se situent très au-dessus, illustrant la montée en compute nécessaire pour les modèles frontier.


In [ ]:
# Coût vs compute (pour les lignes avec coût)
cost_avail = combined.dropna(subset=["compute_flop", "train_cost_usd"])
fig, ax = plt.subplots()
for seg, df_seg in cost_avail.groupby("segment"):
    ax.scatter(df_seg["compute_flop"], df_seg["train_cost_usd"], label=seg, alpha=0.9)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Compute (FLOP)")
ax.set_ylabel("Coût d'entraînement (USD 2023)")
ax.set_title("Coût vs compute")
ax.legend()
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

cost_table = cost_avail.sort_values("train_cost_usd", ascending=False)[["Model", "Organization", "compute_flop", "train_cost_usd", "segment"]]
cost_table.head(10)


Les Qwen n'apparaissent pas ici faute de coûts renseignés. Les modèles Grok4/GPT‑4.5/Llama4 Behemoth/Gemini 1.0 ont des coûts de plusieurs dizaines à centaines de millions USD.


In [ ]:
# Energie estimée (puissance * heures)
energy_df = combined.dropna(subset=["power_w", "train_hours"])
energy_df["energy_mwh"] = energy_df["power_w"] * energy_df["train_hours"] / 1e6

fig, ax = plt.subplots()
for seg, df_seg in energy_df.groupby("segment"):
    ax.bar(df_seg["Model"], df_seg["energy_mwh"], label=seg, color="#4C72B0")
ax.set_ylabel("Energie (MWh)")
ax.set_title("Energie d'entraînement (disponible)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

energy_df[["Model", "Organization", "energy_mwh", "power_w", "train_hours", "segment"]]


Seule Gemini 1.0 Ultra expose puissance et durée dans les CSV : ~48000 MWh estimés. Aucune donnée d'énergie pour Qwen2.5/3 dans ces fichiers.


## Conclusion
- **Taille** : Qwen2.5/3 sont beaucoup plus compacts que Grok4, GPT‑4.5, Llama 4 Behemoth, Gemini 1.x.
- **Compute** : les Big mobilisent 1e25–1e26 FLOP; les entrées Qwen listées n'ont pas de compute renseigné ici, mais sont attendues bien plus bas.
- **Coût** : seules les entrées Big ont des coûts (dizaines/centaines de M$). Absence de coûts Qwen -> à compléter via sources externes si nécessaire.
- **Énergie** : seule Gemini 1.0 Ultra fournit puissance+durée (~48 GWh estimés) ; absence pour Qwen2.5/3.
- **Performance** : non disponible; si besoin de comparer qualité vs coût/compute, intégrer des benchmarks publics (MT-Bench, MMLU, etc.).
- **À ajouter si trouvable** : prix d'inférence, latence, consommation en production, context window, et coûts cloud vs on-prem pour affiner la comparaison opérationnelle.
